# AI News Agents — model-change tracker + legal subagent

This notebook demonstrates an agent that watches **AI provider web pages** for model changes (launches, deprecations, pricing, API changes) and a **legal subagent** that watches terms of service, usage policies and EU AI Act pages affecting LLM providers. Both produce a cited digest saved as Markdown (emailed later).

## Design: deterministic pipeline, LLM only where judgment is needed

```
 pages ─► 1 Fetch ─► 2 Normalize ─► 3 Change checks ─► 4 Classifier ─► 5 Triage (cheap LLM)
          ETag/304                   hash · new lines     skips changes it     large changes only;
          all pages at once          text and DOM ratio   is sure don't        small ones go
                                     keywords             matter               straight to 6
                                                                                 │ meaningful?
                                                                                 ▼
        9 Save ◄─ 8 Digest (template) ◄─ 7 Dedupe ◄─ 6 Extract structured events (LLM)
                                                      new changelog entries, or the line diff
                                                      + verbatim-quote guard
```

What keeps it fast and cheap:

* **Concurrency.** All pages are fetched together, and the model calls for one step run together.
* **Small model calls.** Extraction uses Flash, thinking is off for triage and capped for extraction, and the schemas ask for short fields.
* **Fewer model calls.** Small changes skip triage, a classifier trained on past verdicts skips changes that clearly don't matter, identical requests are answered from a disk cache, and the digest is a template.
* **Less input.** Changelog pages send only the new or edited entries, each with its date heading.
* **Batch mode** (optional) sends each step as a Vertex AI batch job: about half the price, but slower.

## Setup

Auth uses Google Application Default Credentials — run once in a terminal, then set the values in the config cell below:

```
gcloud auth application-default login
```

In [ ]:
# %pip install -r requirements.txt

# All configuration lives here. Edit these values directly.

# Vertex AI
PROJECT_ID = "your-gcp-project"
REGION = "global"

# Gemini models, swappable per step.
# Check Vertex Model Garden for the model versions enabled in your project.
TRIAGE_MODEL = "gemini-2.5-flash-lite"
EXTRACT_MODEL = "gemini-2.5-flash"   # copying facts out of a diff doesn't need Pro; the quote guard checks the output

# Thinking-token budget per step. Thinking is billed as output and is most of a call's latency.
# 0 turns thinking off (Flash and Flash-Lite only; Pro needs at least 128).
THINKING_BUDGET = {"triage": 0, "redesign": 0, "extract": 1024}

# USD per 1M tokens (input, output), for the cost estimates in section 12. Thinking tokens count as output.
# Check current prices: https://cloud.google.com/vertex-ai/generative-ai/pricing (Pro: prompts up to 200k tokens)
MODEL_PRICES = {
    "gemini-2.5-flash-lite": (0.10, 0.40),
    "gemini-2.5-flash": (0.30, 2.50),
    "gemini-2.5-pro": (1.25, 10.00),
}

# HTTP
HTTP_TIMEOUT_SECONDS = 30
USER_AGENT = "ai-news-agents-demo/0.1 (change monitoring)"
MAX_CONCURRENT = 8   # page fetches and model calls in flight at once

# Paths and limits
SNAPSHOT_DIR = "snapshots"
BASELINE_DIR = "baselines"   # seeded older copies, used when a source has no snapshot yet
OUTPUT_DIR = "output"
TRIAGE_LOG = "output/triage_log.jsonl"   # every model verdict + its signals: training data for the classifier
LLM_CACHE_DIR = "cache/llm"              # model replies by request hash: an identical request is never paid twice
MAX_DIFF_CHARS = 20_000

# Change checks (section 5). Cleaning strips these patterns from every line before comparing.
# Keep them narrow: a pattern that matches real content hides real changes.
IGNORE_PATTERNS = [
    # "Last updated: October 28, 2026" / "Updated 2026-10-28" (date-only lines)
    r"^(last )?(updated|modified|reviewed)( on)?:?\s+(\w+ \d{1,2},? \d{4}|\d{1,2} \w+ \d{4}|\d{4}-\d{2}-\d{2})\.?$",
    r"\b\d+\s*min(ute)?s? read\b",                                         # "5 min read"
    r"^(©|copyright\s*©?)\s*(\d{4}\s*[-–]\s*)?\d{4}\b.*$",                   # "© 2026 Example Inc."
    r"^(?=.{0,200}$)(?=.*\bcookies?\b)(?=.*\b(accept|reject|consent|preferences|settings)\b).*$",  # cookie banners
    r"^(was this page helpful\??|share|print|back to top|skip to (main )?content|on this page)$",
]
BROKEN_RATIO = 0.3   # cleaned text shrank below 30% of the previous copy: extraction probably broke
DRASTIC_TEXT = 0.5   # more than half the text changed...
DRASTIC_DOM = 0.5    # ...and more than half the page's tag structure changed: likely a redesign

# Changelog pages (at least this many date headings) send whole new or edited entries to extraction
# instead of a line diff, so every change arrives with its date.
CHANGELOG_MIN_DATE_HEADINGS = 3

# Changes with new lines up to this size skip triage and go straight to extraction
# (an empty event list then means "not meaningful"). Bigger ones are triaged by the cheap model first.
DIRECT_EXTRACT_MAX_CHARS = 4_000

# Triage classifier (section 8), trained on TRIAGE_LOG. It only ever skips the model for changes it is
# sure don't matter; everything else still goes to the model.
CLASSIFIER_MIN_EXAMPLES = 50     # labelled log rows needed (at least 5 of each label) before training
CLASSIFIER_SKIP_BELOW = 0.05     # skip the model when P(meaningful) is below this
CLASSIFIER_MAX_MISS_RATE = 0.01  # use the classifier only if cross-validation says it skips ≤1% of real changes

# Batch mode: send each step's model calls as one Vertex AI batch job per model. About 50% cheaper, but a job
# can take minutes to hours, so use it for scheduled runs. Needs a Cloud Storage bucket the project can write to.
USE_BATCH = False
BATCH_GCS_PREFIX = "gs://your-bucket/ai-news-agents/batches"
BATCH_DISCOUNT = 0.5
BATCH_POLL_SECONDS = 30

# Savings estimate (section 12) when a run made no triage calls to measure
FALLBACK_TRIAGE_SECONDS = 2.0
FALLBACK_TRIAGE_USD = 0.0005

## Sources

Primary sources only: provider docs, changelogs, pricing and policy pages, and official EU pages. Every source is a web page and goes through the same checks.

In [ ]:
MODEL_SOURCES = [
    {"name": "Anthropic model deprecations", "provider": "Anthropic",
     "url": "https://docs.anthropic.com/en/docs/about-claude/model-deprecations"},
    {"name": "Anthropic API release notes", "provider": "Anthropic",
     "url": "https://docs.anthropic.com/en/release-notes/api"},
    {"name": "Gemini API changelog", "provider": "Google",
     "url": "https://ai.google.dev/gemini-api/docs/changelog"},
    {"name": "Google AI Studio pricing", "provider": "Google",
     "url": "https://aistudio.google.com/docs/pricing"},
    {"name": "Vertex AI generative AI release notes", "provider": "Google",
     "url": "https://cloud.google.com/vertex-ai/generative-ai/docs/release-notes"},
    {"name": "Mistral changelog", "provider": "Mistral",
     "url": "https://docs.mistral.ai/getting-started/changelog/"},
]

LEGAL_SOURCES = [
    {"name": "Anthropic usage policy", "provider": "Anthropic",
     "url": "https://www.anthropic.com/legal/aup"},
    {"name": "OpenAI usage policies", "provider": "OpenAI",
     "url": "https://openai.com/policies/usage-policies/"},
    {"name": "EU AI Act regulatory framework", "provider": "European Commission",
     "url": "https://digital-strategy.ec.europa.eu/en/policies/regulatory-framework-ai"},
    {"name": "EU GPAI Code of Practice", "provider": "European Commission",
     "url": "https://digital-strategy.ec.europa.eu/en/policies/contents-code-gpai"},
]

## Event schemas

The LLM must return these shapes (structured output), so downstream code never parses free text. Output tokens are
most of a call's latency, so the descriptions ask for short fields, and `source_url` is left out of the schema the
model sees: the pipeline fills it in.

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field
from pydantic.json_schema import SkipJsonSchema


class Triage(BaseModel):
    meaningful: bool = Field(description="False for layout, navigation, typo or boilerplate-only changes")
    category: str
    severity: Literal["low", "medium", "high"]
    reason: str = Field(description="At most 15 words")


class ChangeEvent(BaseModel):
    provider: str
    model: str | None
    change_type: Literal["new_model", "deprecation", "retirement", "pricing", "api_change",
                         "rate_limit", "capability", "other"]
    effective_date: str | None = Field(description="ISO date if the page states one")
    summary: str = Field(description="One sentence, at most 25 words")
    quote: str = Field(description="One sentence or table row copied verbatim from the NEW page text")
    source_url: SkipJsonSchema[str] = ""   # hidden from the model; set by the pipeline


class LegalEvent(BaseModel):
    jurisdiction: Literal["EU", "US-federal", "US-state", "UK", "other"]
    instrument: Literal["legislation", "regulation", "guidance", "terms_of_service",
                        "litigation", "enforcement", "other"]
    title: str
    parties: list[str] = Field(description="Companies, agencies or courts involved")
    effective_date: str | None
    summary: str = Field(description="One sentence, at most 25 words")
    impact: str = Field(description="One sentence: the practical effect on companies providing LLM services")
    quote: str = Field(description="One sentence copied verbatim from the NEW page text")
    source_url: SkipJsonSchema[str] = ""   # hidden from the model; set by the pipeline


class ChangeEvents(BaseModel):
    events: list[ChangeEvent]


class LegalEvents(BaseModel):
    events: list[LegalEvent]

## Fetch + normalize

HTML → main text (trafilatura strips nav and footers, keeps tables). JavaScript-app pages that ship their content as text in `<noscript><pre>` (such as the AI Studio docs) use that copy instead, because it keeps its line breaks. `fetch()` sends the `ETag` / `Last-Modified` values saved by the previous run, so an unchanged page can answer `304 Not Modified` without being downloaded.

Fetching is async, and `LIMIT` caps how many requests (page fetches and model calls) are in flight at once.
Jupyter runs top-level `await` directly.

In [ ]:
import asyncio
import time
from dataclasses import dataclass

import httpx
import lxml.etree
import lxml.html
import trafilatura

HTTP = httpx.AsyncClient(follow_redirects=True, timeout=HTTP_TIMEOUT_SECONDS, headers={"User-Agent": USER_AGENT})
LIMIT = asyncio.Semaphore(MAX_CONCURRENT)   # shared by page fetches and model calls


@dataclass
class Fetched:
    html: str | None          # None when the server answered 304 Not Modified
    etag: str | None
    last_modified: str | None
    seconds: float


async def fetch(url: str, validators: dict | None = None) -> Fetched:
    validators = validators or {}
    headers = {}
    if validators.get("etag"):
        headers["If-None-Match"] = validators["etag"]
    if validators.get("last_modified"):
        headers["If-Modified-Since"] = validators["last_modified"]
    async with LIMIT:
        started = time.perf_counter()
        resp = await HTTP.get(url, headers=headers)
        seconds = time.perf_counter() - started
    if resp.status_code == 304:
        return Fetched(None, validators.get("etag"), validators.get("last_modified"), seconds)
    resp.raise_for_status()
    return Fetched(resp.text, resp.headers.get("etag"), resp.headers.get("last-modified"), seconds)


def _noscript_text(html: str) -> str:
    """JS-app pages (e.g. AI Studio docs) ship their content as source text in <noscript><pre>."""
    try:
        root = lxml.html.fromstring(html)
    except (lxml.etree.ParserError, ValueError):
        return ""
    return "\n".join(pre.text_content() for pre in root.xpath("//noscript//pre"))


def normalize(html: str) -> str:
    text = trafilatura.extract(html, include_tables=True, include_links=False, favor_recall=True) or ""
    # trafilatura flattens <pre> line breaks, so prefer the no-JS copy when it holds more of the page
    fallback = _noscript_text(html)
    return (fallback if len(fallback) > len(text) else text).strip()


demo_src = MODEL_SOURCES[0]
try:
    demo_page = await fetch(demo_src["url"])
    demo_text = normalize(demo_page.html)
    print(f"{demo_src['name']}: {len(demo_text):,} chars in {demo_page.seconds:.2f}s (ETag: {demo_page.etag})\n")
    print(demo_text[:1500])
except httpx.HTTPError as e:
    print(f"Could not fetch {demo_src['url']}: {e}")

## Snapshots + change checks

Each source keeps three files in `snapshots/<domain>/`: the normalized text (`.txt`), the raw HTML (`.html`, for
the DOM check) and the HTTP validators (`.meta.json`, for the next `304`). When a source has no snapshot yet, the run
compares against its **seeded baseline** in `baselines/<domain>/<source>.md`: the source's text with its newest entries
removed, so the first run reports real, recent changes. Baselines are text only, so the DOM check is skipped that time.

Every page goes through all of these checks before any model is called:

| Check | Outcome | What happens |
|---|---|---|
| `304 Not Modified`, or the cleaned texts hash the same | `none` | no change |
| Cleaned text shrank below `BROKEN_RATIO` of the previous copy | `broken` | flagged; the previous snapshot is kept; no LLM |
| No new lines (only removed, reordered or layout changes) | `layout` | ignored |
| Text changed more than `DRASTIC_TEXT` **and** tag structure changed more than `DRASTIC_DOM` | `drastic` | cheap model: redesign, or did a tracked fact change? |
| New lines, but none match the domain's `keywords` | `no_keywords` | ignored |
| Otherwise | `normal` | cheap-model triage on the new lines only |

* **Cleaning** collapses whitespace and strips text matching `IGNORE_PATTERNS` (date-only lines, read times,
  copyright lines, cookie banners).
* **New lines** are lines of the new copy that appear nowhere in the old one, so moved lines don't count.
* **Text ratio** is `1 − difflib` similarity over the cleaned lines (0 = identical, 1 = nothing in common).
* **DOM ratio** is `1 − Jaccard` similarity of the pages' tag paths (`html>body>main>table>tr`), a cheap fingerprint of
  the page template.

In [ ]:
import difflib
import hashlib
import json
import re
from pathlib import Path


def _slug(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", "-", name.lower()).strip("-")


def _snapshot_path(domain: str, name: str, ext: str = "txt") -> Path:
    return Path(SNAPSHOT_DIR) / domain / f"{_slug(name)}.{ext}"


def _read(path: Path) -> str | None:
    return path.read_text() if path.exists() else None


def load_snapshot(domain: str, name: str) -> str | None:
    return _read(_snapshot_path(domain, name))


def load_snapshot_html(domain: str, name: str) -> str | None:
    return _read(_snapshot_path(domain, name, "html"))


def load_meta(domain: str, name: str) -> dict:
    raw = _read(_snapshot_path(domain, name, "meta.json"))
    return json.loads(raw) if raw else {}


def load_baseline(domain: str, name: str) -> str | None:
    return _read(Path(BASELINE_DIR) / domain / f"{_slug(name)}.md")


def save_snapshot(domain: str, name: str, text: str, page: Fetched) -> None:
    files = {"txt": text, "html": page.html,
             "meta.json": json.dumps({"etag": page.etag, "last_modified": page.last_modified})}
    for ext, content in files.items():
        path = _snapshot_path(domain, name, ext)
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(content)


def _squash(s: str) -> str:
    return re.sub(r"\s+", " ", s).strip().lower()


def clean_for_diff(text: str) -> str:
    """Collapse whitespace and strip volatile text (IGNORE_PATTERNS) so it can't register as a change."""
    lines = []
    for line in text.splitlines():
        line = re.sub(r"\s+", " ", line).strip()
        for pattern in IGNORE_PATTERNS:
            line = re.sub(pattern, "", line, flags=re.IGNORECASE).strip()
        if line:
            lines.append(line)
    return "\n".join(lines)


def diff_text(old: str, new: str) -> str:
    lines = difflib.unified_diff(old.splitlines(), new.splitlines(), "before", "after", n=2, lineterm="")
    return "\n".join(lines)


_NON_CONTENT = "//head|//script|//style|//noscript|//template|//svg"


def tag_paths(html: str | None) -> set[str]:
    """Distinct tag chains such as 'html>body>main>table>tr': a cheap fingerprint of the page template."""
    if not html:
        return set()
    try:
        root = lxml.html.fromstring(html)
    except (lxml.etree.ParserError, ValueError):
        return set()
    for el in root.xpath(_NON_CONTENT):
        el.drop_tree()
    return {">".join([a.tag for a in reversed(list(el.iterancestors()))] + [el.tag])
            for el in root.iter() if isinstance(el.tag, str)}


def dom_change_ratio(old_html: str | None, new_html: str | None) -> float:
    old, new = tag_paths(old_html), tag_paths(new_html)
    if not old or not new:
        return 0.0   # nothing to compare (e.g. a text-only seeded baseline): treat structure as unchanged
    return 1 - len(old & new) / len(old | new)


@dataclass
class Signals:
    old_clean: str
    new_clean: str
    added: list[str]       # lines in the new copy that appear nowhere in the old one
    removed: list[str]     # lines in the old copy that appear nowhere in the new one
    text_change: float     # 0 = identical lines, 1 = nothing in common
    dom_change: float      # 0 = same tag structure, 1 = nothing in common
    length_ratio: float    # new cleaned length / old cleaned length

    def summary(self) -> dict:
        return {"added_lines": len(self.added), "removed_lines": len(self.removed),
                "text_change": round(self.text_change, 3), "dom_change": round(self.dom_change, 3),
                "length_ratio": round(self.length_ratio, 3)}


def _hash(text: str) -> bytes:
    return hashlib.sha256(text.encode()).digest()


def change_signals(old_text: str, new_text: str, old_html: str | None, new_html: str | None) -> Signals:
    old_clean, new_clean = clean_for_diff(old_text), clean_for_diff(new_text)
    if _hash(old_clean) == _hash(new_clean):   # the common case: skip the diffs entirely
        return Signals(old_clean, new_clean, [], [], 0.0, 0.0, 1.0)
    old_lines, new_lines = old_clean.splitlines(), new_clean.splitlines()
    old_set, new_set = {_squash(l) for l in old_lines}, {_squash(l) for l in new_lines}
    return Signals(
        old_clean, new_clean,
        added=[l for l in new_lines if _squash(l) not in old_set],
        removed=[l for l in old_lines if _squash(l) not in new_set],
        text_change=1 - difflib.SequenceMatcher(None, old_lines, new_lines).ratio(),
        dom_change=dom_change_ratio(old_html, new_html),
        length_ratio=len(new_clean) / max(len(old_clean), 1),
    )


def classify_change(sig: Signals, keywords: str) -> str:
    """none | broken | layout | drastic | no_keywords | normal (see the table above)."""
    if _hash(sig.old_clean) == _hash(sig.new_clean):
        return "none"
    if sig.length_ratio < BROKEN_RATIO:
        return "broken"
    if not sig.added:
        return "layout"
    if sig.text_change > DRASTIC_TEXT and sig.dom_change > DRASTIC_DOM:
        return "drastic"
    if not re.search(keywords, "\n".join(sig.added), re.IGNORECASE):
        return "no_keywords"
    return "normal"

### Cleaning example

What `clean_for_diff` removes, first on a made-up snippet, then on a real seeded baseline. The cleaned real file is
also written to `output/` so you can open it next to the original.

In [ ]:
sample = """Model deprecations
Last updated: October 28, 2026
   Example Model 1      retires on   February 19, 2027
5 min read
We use cookies to improve your experience. Accept all cookies
© 2026 Example AI Inc. All rights reserved."""
print("BEFORE\n------\n" + sample + "\n\nAFTER\n-----\n" + clean_for_diff(sample))

real_name = "Anthropic model deprecations"
real = load_baseline("models", real_name) or ""
cleaned = clean_for_diff(real)
kept = {_squash(l) for l in cleaned.splitlines()}
dropped = [l.strip() for l in real.splitlines() if l.strip() and _squash(l) not in kept]
print(f"\n{real_name} baseline: {len(real):,} → {len(cleaned):,} chars; "
      f"{len(dropped)} lines removed or trimmed{':' if dropped else ''}")
for line in dropped[:20]:
    print("  -", line)

cleaned_path = Path(OUTPUT_DIR) / f"cleaned-{_slug(real_name)}.txt"
cleaned_path.parent.mkdir(parents=True, exist_ok=True)
cleaned_path.write_text(cleaned)
print(f"\ncleaned file: {cleaned_path}  (original: {Path(BASELINE_DIR) / 'models' / (_slug(real_name) + '.md')})")

### Check examples

Made-up before/after pairs run through `change_signals` + `classify_change`, one per outcome.

In [ ]:
import pandas as pd

BASE_TEXT = ("Model deprecations\nExample Model 1 retires on 2027-02-19\n"
             "Example Model 2 retires on 2027-06-01\nUse the migration guide to upgrade.")
BASE_HTML = "<html><body><main><h1>x</h1><table><tr><td>x</td></tr></table><p>x</p></main></body></html>"
REDESIGN_HTML = "<html><body><div><section><ul><li>x</li></ul></section><footer>x</footer></div></body></html>"
lines = BASE_TEXT.splitlines()

examples = {
    "whitespace only": (BASE_TEXT.replace(" ", "   "), BASE_HTML),
    "'Last updated' date added": ("Last updated: November 4, 2026\n" + BASE_TEXT, BASE_HTML),
    "layout tweak, same text": (BASE_TEXT, BASE_HTML.replace("<main>", "<main><div>").replace("</main>", "</div></main>")),
    "lines reordered": ("\n".join([lines[0], lines[2], lines[1], lines[3]]), BASE_HTML),
    "extraction broke": (lines[0], BASE_HTML),
    "redesign": ("Deprecated models\nExample Model 1: retirement date 2027-02-19\n"
                 "Example Model 2: retirement date 2027-06-01\nSee the upgrade guide.", REDESIGN_HTML),
    "new line, no keywords": (BASE_TEXT + "\nJoin our community forum.", BASE_HTML),
    "new deprecation row": (BASE_TEXT + "\nExample Model 3 retires on 2027-09-01", BASE_HTML),
}
rows = []
for label, (text, html) in examples.items():
    sig = change_signals(BASE_TEXT, text, BASE_HTML, html)
    rows.append({"example": label, "outcome": classify_change(sig, r"retir|deprecat|pric"), **sig.summary()})
pd.DataFrame(rows)

### Changelog entries

Changelogs are lists of entries under date headings, and most changes are a new entry at the top. When a page has at
least `CHANGELOG_MIN_DATE_HEADINGS` date headings, extraction gets the whole new or edited entries instead of a line
diff. New lines are marked `+`, so the model knows what changed, and each change arrives with its date. Removed lines
and overlapping diff context are left out. Other pages (pricing tables, policies) still get the line diff.

In [ ]:
MONTH = r"(jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)[a-z]*\.?"
# A line that starts with a date ("October 23, 2026", "September 29", "4 November 2026", "2026-06-05")
# and is short enough to be a heading, such as "2026-06-05: Claude Opus 4.1 model"
DATE_HEADING = re.compile(
    rf"^#*\s*({MONTH} \d{{1,2}}(,? \d{{4}})?|\d{{1,2}} {MONTH}( \d{{4}})?|\d{{4}}-\d{{2}}-\d{{2}})\b.{{0,60}}$",
    re.IGNORECASE,
)


def is_changelog(text: str) -> bool:
    return sum(bool(DATE_HEADING.match(line)) for line in text.splitlines()) >= CHANGELOG_MIN_DATE_HEADINGS


def split_entries(text: str) -> list[list[str]]:
    """Split a page into entries, each starting at a date heading (text before the first heading is one entry)."""
    entries = [[]]
    for line in text.splitlines():
        if DATE_HEADING.match(line) and entries[-1]:
            entries.append([])
        entries[-1].append(line)
    return [e for e in entries if e]


def changed_entries(sig: Signals) -> str:
    """The entries of the new copy that contain new lines, with those lines marked '+ '."""
    added = {_squash(line) for line in sig.added}
    blocks = []
    for entry in split_entries(sig.new_clean):
        if any(_squash(line) in added for line in entry):
            blocks.append("\n".join(("+ " if _squash(line) in added else "  ") + line for line in entry))
    return "\n\n".join(blocks)


def extraction_input(sig: Signals) -> str:
    """What extraction sees: new or edited entries for changelogs, the line diff for everything else."""
    if is_changelog(sig.new_clean):
        return changed_entries(sig)
    return diff_text(sig.old_clean, sig.new_clean)


for domain_dir in sorted(Path(BASELINE_DIR).iterdir()):
    for baseline in sorted(domain_dir.glob("*.md")):
        text = clean_for_diff(baseline.read_text())
        print(f"{domain_dir.name}/{baseline.stem}: {'changelog' if is_changelog(text) else 'other page'} "
              f"({len(split_entries(text))} entries)")

old_log = "Changelog\nOctober 21, 2026\nExample Model 1 is deprecated.\nOctober 20, 2026\nNew batch endpoint."
new_log = ("Changelog\nOctober 23, 2026\nExample Model 3 is now available.\nOctober 21, 2026\n"
           "Example Model 1 is deprecated and retires on 2027-01-15.\nOctober 20, 2026\nNew batch endpoint.")
print("\n" + changed_entries(change_signals(old_log, new_log, None, None)))

## LLM calls (Gemini on Vertex AI)

Each model call is an `LLMRequest`: the step, the model, the prompt and the pydantic schema of the answer. The
`google-genai` client uses `vertexai=True`, structured JSON output and the step's thinking budget from
`THINKING_BUDGET`.

`ask_all(requests)` answers a list of requests and returns the parsed answers in the same order:

1. Requests answered before (same model, settings and prompt) come from the disk cache in `LLM_CACHE_DIR`, for free.
2. The rest run concurrently (at most `MAX_CONCURRENT` at a time), or as batch jobs when `USE_BATCH` is on.

Every answer is appended to `LLM_CALLS` with its source, latency, token counts, estimated cost and `via`
(`live`, `batch` or `cache`).

Prompts put the instructions first and the page text last. Gemini's implicit cache bills a repeated prompt prefix at a
lower rate once that prefix is long enough (about 1–2k tokens), so instructions that grow keep paying off.

In [ ]:
from dataclasses import asdict
from functools import cache

from google.genai import types
from pydantic import ValidationError

SYSTEM = ("You analyze changes to web pages for a monitoring service. Use only facts present in the input; "
          "never infer dates, prices or model names that are not written there.")

LLM_CALLS: list[dict] = []   # one row per answered request


@dataclass
class LLMRequest:
    domain: str
    source: str
    step: str                   # triage | redesign | extract (a key of THINKING_BUDGET)
    model: str
    prompt: str
    schema: type[BaseModel]


@dataclass
class Reply:
    text: str
    tokens_in: int
    tokens_out: int             # visible output + thinking tokens (both billed as output)


def response_schema(schema: type[BaseModel]) -> dict:
    """The pydantic model as plain JSON Schema, with every $ref replaced by its definition."""
    raw = schema.model_json_schema()
    definitions = raw.pop("$defs", {})

    def inline(node):
        if isinstance(node, dict):
            if "$ref" in node:
                return inline(definitions[node["$ref"].split("/")[-1]])
            return {key: inline(value) for key, value in node.items()}
        if isinstance(node, list):
            return [inline(value) for value in node]
        return node

    return inline(raw)


@cache
def _gemini():
    from google import genai
    return genai.Client(vertexai=True, project=PROJECT_ID, location=REGION)


def _config(req: LLMRequest) -> types.GenerateContentConfig:
    return types.GenerateContentConfig(
        system_instruction=SYSTEM,
        response_mime_type="application/json",
        response_json_schema=response_schema(req.schema),
        thinking_config=types.ThinkingConfig(thinking_budget=THINKING_BUDGET[req.step]),
    )


# Response cache

def _cache_path(req: LLMRequest) -> Path:
    settings = {"model": req.model, "system": SYSTEM, "schema": response_schema(req.schema),
                "thinking_budget": THINKING_BUDGET[req.step], "prompt": req.prompt}
    key = hashlib.sha256(json.dumps(settings, sort_keys=True).encode()).hexdigest()
    return Path(LLM_CACHE_DIR) / f"{key}.json"


def load_cached(req: LLMRequest) -> Reply | None:
    raw = _read(_cache_path(req))
    return Reply(**json.loads(raw)) if raw else None


def save_cached(req: LLMRequest, reply: Reply) -> None:
    path = _cache_path(req)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(asdict(reply)))


# Metering

def record(req: LLMRequest, reply: Reply, latency_s: float, via: str) -> None:
    """Log one answered request. Cache hits cost nothing; batch calls get the batch discount."""
    price_in, price_out = MODEL_PRICES.get(req.model, (0.0, 0.0))
    cost = (reply.tokens_in * price_in + reply.tokens_out * price_out) / 1e6
    if via == "cache":
        cost = 0.0
    if via == "batch":
        cost *= BATCH_DISCOUNT
    LLM_CALLS.append({"domain": req.domain, "source": req.source, "step": req.step, "model": req.model,
                      "via": via, "latency_s": latency_s, "tokens_in": reply.tokens_in,
                      "tokens_out": reply.tokens_out, "cost_usd": cost})


# Live calls

def _reply_from(resp: types.GenerateContentResponse) -> Reply:
    usage = resp.usage_metadata
    tokens_in = (usage.prompt_token_count or 0) if usage else 0
    tokens_out = ((usage.candidates_token_count or 0) + (usage.thoughts_token_count or 0)) if usage else 0
    return Reply(resp.text or "", tokens_in, tokens_out)


async def call_live(req: LLMRequest) -> Reply:
    async with LIMIT:
        started = time.perf_counter()
        resp = await _gemini().aio.models.generate_content(model=req.model, contents=req.prompt, config=_config(req))
    reply = _reply_from(resp)
    record(req, reply, time.perf_counter() - started, via="live")
    return reply


def parse(req: LLMRequest, reply: Reply | None) -> BaseModel | None:
    if reply is None:
        print(f"  {req.source}: no {req.step} answer from {req.model}; skipping")
        return None
    try:
        return req.schema.model_validate_json(reply.text)
    except ValidationError:
        print(f"  {req.source}: {req.model} returned no parseable {req.step} output; skipping")
        return None

### Batch mode and `ask_all`

With `USE_BATCH = True`, the requests that missed the cache are written to Cloud Storage as JSONL (one Vertex AI batch
job per model), and the run waits for the jobs and reads the predictions back. Batch output isn't guaranteed to keep
the input order, so replies are matched to requests by prompt (each prompt names its source, so prompts are unique).
Every request in a job is logged with the job's full wall-clock time, because that is how long it waited.

In [ ]:
from collections import defaultdict
from datetime import datetime, timezone

BATCH_FINISHED = {types.JobState.JOB_STATE_SUCCEEDED, types.JobState.JOB_STATE_PARTIALLY_SUCCEEDED,
                  types.JobState.JOB_STATE_FAILED, types.JobState.JOB_STATE_CANCELLED,
                  types.JobState.JOB_STATE_EXPIRED}


def _gcs_blob(uri: str):
    from google.cloud import storage
    bucket, _, name = uri.removeprefix("gs://").partition("/")
    return storage.Client(project=PROJECT_ID).bucket(bucket).blob(name)


def _gcs_jsonl(prefix_uri: str, filename: str) -> list[dict]:
    """Every JSON line of every file called `filename` under a gs:// prefix."""
    from google.cloud import storage
    bucket, _, prefix = prefix_uri.removeprefix("gs://").partition("/")
    rows = []
    for blob in storage.Client(project=PROJECT_ID).list_blobs(bucket, prefix=prefix):
        if blob.name.endswith(filename):
            rows += [json.loads(line) for line in blob.download_as_text().splitlines() if line.strip()]
    return rows


def _batch_line(req: LLMRequest) -> dict:
    """One request in the JSONL format that Vertex AI batch prediction reads (the REST form of `_config`)."""
    return {"request": {
        "contents": [{"role": "user", "parts": [{"text": req.prompt}]}],
        "systemInstruction": {"parts": [{"text": SYSTEM}]},
        "generationConfig": {"responseMimeType": "application/json",
                             "responseJsonSchema": response_schema(req.schema),
                             "thinkingConfig": {"thinkingBudget": THINKING_BUDGET[req.step]}},
    }}


def _reply_from_json(response: dict) -> Reply:
    parts = response.get("candidates", [{}])[0].get("content", {}).get("parts", [])
    usage = response.get("usageMetadata", {})
    return Reply(text="".join(p.get("text", "") for p in parts if not p.get("thought")),
                 tokens_in=usage.get("promptTokenCount", 0),
                 tokens_out=usage.get("candidatesTokenCount", 0) + usage.get("thoughtsTokenCount", 0))


async def run_batch_job(model: str, requests: list[LLMRequest]) -> list[Reply | None]:
    """Run one batch job and return the replies in request order (None where a request failed)."""
    job_dir = f"{BATCH_GCS_PREFIX.rstrip('/')}/{datetime.now(timezone.utc):%Y%m%dT%H%M%S}-{_slug(model)}"
    _gcs_blob(f"{job_dir}/input.jsonl").upload_from_string(
        "\n".join(json.dumps(_batch_line(r)) for r in requests), content_type="application/jsonl")

    started = time.perf_counter()
    job = await _gemini().aio.batches.create(model=model, src=f"{job_dir}/input.jsonl",
                                             config=types.CreateBatchJobConfig(dest=f"{job_dir}/output"))
    print(f"  batch job {job.name}: {len(requests)} {model} requests")
    while job.state not in BATCH_FINISHED:
        await asyncio.sleep(BATCH_POLL_SECONDS)
        job = await _gemini().aio.batches.get(name=job.name)
    waited = time.perf_counter() - started
    print(f"  batch job {job.name}: {job.state.name} after {waited:.0f}s")

    by_prompt = {row["request"]["contents"][0]["parts"][0]["text"]: _reply_from_json(row["response"])
                 for row in _gcs_jsonl(job.dest.gcs_uri, "predictions.jsonl") if row.get("response")}
    replies = [by_prompt.get(r.prompt) for r in requests]
    for req, reply in zip(requests, replies):
        if reply:
            record(req, reply, waited, via="batch")
    return replies


async def run_batches(requests: list[LLMRequest]) -> list[Reply | None]:
    """One batch job per model, all running at once; replies come back in request order."""
    by_model: dict[str, list[int]] = defaultdict(list)
    for i, req in enumerate(requests):
        by_model[req.model].append(i)
    jobs = [run_batch_job(model, [requests[i] for i in indexes]) for model, indexes in by_model.items()]
    replies: list[Reply | None] = [None] * len(requests)
    for indexes, job_replies in zip(by_model.values(), await asyncio.gather(*jobs)):
        for i, reply in zip(indexes, job_replies):
            replies[i] = reply
    return replies


async def ask_all(requests: list[LLMRequest]) -> list[BaseModel | None]:
    """Answer every request (from the cache, live or by batch) and return the parsed answers in order."""
    replies = [load_cached(req) for req in requests]
    for req, reply in zip(requests, replies):
        if reply:
            record(req, reply, 0.0, via="cache")

    missing = [i for i, reply in enumerate(replies) if reply is None]
    to_send = [requests[i] for i in missing]
    if USE_BATCH:
        fresh = await run_batches(to_send) if to_send else []
    else:
        fresh = await asyncio.gather(*(call_live(req) for req in to_send))
    for i, reply in zip(missing, fresh):
        replies[i] = reply

    answers = [parse(req, reply) for req, reply in zip(requests, replies)]
    for i in missing:
        if answers[i] is not None:   # cache only answers that parsed
            save_cached(requests[i], replies[i])
    return answers

## Triage + extract (with the quote guard)

Only pages the checks can't settle reach a model, and these functions build the requests for them. `triage_request`
sends just the new lines (plus a few removed lines for context). `redesign_request` handles the `drastic` case: it
compares removed and added lines and asks whether a tracked fact changed or only the wording and layout did.
`extract_request` turns the changes into structured events (see `extraction_input` above).

Every verdict is logged to `TRIAGE_LOG` with the check signals, as training data for the classifier. Changes that skip
triage log their extraction result instead: no events means not meaningful. `keep_verbatim` drops any event whose
quote isn't found in the new page text, and sets each event's `source_url`.

In [ ]:
def _clip(text: str, limit: int = MAX_DIFF_CHARS) -> str:
    if len(text) <= limit:
        return text
    print(f"  input is {len(text):,} chars; sending the first {limit:,} (raise MAX_DIFF_CHARS to send more)")
    return text[:limit]


# Prompts start with the parts that are the same for every page of a domain and end with the page text.

def triage_request(domain: "Domain", page: "Page") -> LLMRequest:
    added = _clip("\n".join(page.sig.added))
    removed = _clip("\n".join(page.sig.removed), MAX_DIFF_CHARS // 4)
    prompt = (f"We track: {domain.focus}\n"
              "Is this change to a monitored page meaningful for what we track?\n\n"
              f"Source: {page.source['name']} ({page.source['url']})\n\n"
              f"Lines added since the last check:\n<added>\n{added}\n</added>\n\n"
              f"Lines removed (context only):\n<removed>\n{removed}\n</removed>")
    return LLMRequest(domain.name, page.source["name"], "triage", TRIAGE_MODEL, prompt, Triage)


def redesign_request(domain: "Domain", page: "Page") -> LLMRequest:
    sig = page.sig
    removed = _clip("\n".join(sig.removed), MAX_DIFF_CHARS // 2)
    added = _clip("\n".join(sig.added), MAX_DIFF_CHARS // 2)
    prompt = (f"We track: {domain.focus}\n"
              "Both the wording and the structure of this page changed a lot, which is often a site redesign. "
              "Set meaningful=true only if a tracked fact (a model, date, price, limit, rule or obligation) was "
              "added, removed or changed, not if the same facts were just reworded or moved.\n\n"
              f"Source: {page.source['name']} ({page.source['url']})\n"
              f"Wording changed: {sig.text_change:.0%}. Page structure changed: {sig.dom_change:.0%}.\n\n"
              f"Old lines:\n<removed>\n{removed}\n</removed>\n\nNew lines:\n<added>\n{added}\n</added>")
    return LLMRequest(domain.name, page.source["name"], "redesign", TRIAGE_MODEL, prompt, Triage)


def extract_request(domain: "Domain", page: "Page") -> LLMRequest:
    prompt = (f"We track: {domain.focus}\n"
              "List one event per distinct change shown by the new lines (those starting with +). Lines starting "
              "with - were removed; other lines are context. For `quote`, copy the relevant text exactly as it "
              "appears, without the leading +. Return an empty list if nothing relevant changed.\n\n"
              f"Source: {page.source['name']} ({page.source['url']})\n"
              f"Provider/publisher: {page.source['provider']}\n\n"
              f"Changed page text:\n<changes>\n{_clip(extraction_input(page.sig))}\n</changes>")
    return LLMRequest(domain.name, page.source["name"], "extract", EXTRACT_MODEL, prompt, domain.events_schema)


def log_verdict(domain: str, page: "Page", step: str, verdict: dict | None) -> None:
    """Append a verdict (at least {"meaningful": bool}) and the page's signals to TRIAGE_LOG."""
    path = Path(TRIAGE_LOG)
    path.parent.mkdir(parents=True, exist_ok=True)
    record = {"time": datetime.now(timezone.utc).isoformat(timespec="seconds"), "domain": domain,
              "source": page.source["name"], "step": step, **page.sig.summary(), "verdict": verdict,
              "added_text": "\n".join(page.sig.added)[:MAX_DIFF_CHARS]}
    with path.open("a") as f:
        f.write(json.dumps(record) + "\n")


def keep_verbatim(events: list[BaseModel], page: "Page") -> list[BaseModel]:
    """Keep the events whose quote is on the new page, and cite the page (never trust the model with the URL)."""
    kept = []
    for event in events:
        if _squash(event.quote) in _squash(page.new_text):
            event.source_url = page.source["url"]
            kept.append(event)
        else:
            print(f"  {page.source['name']}: dropped event (quote not found in page): {event.summary[:80]!r}")
    return kept

In [ ]:
from typing import Callable


@dataclass
class Domain:
    name: str
    sources: list[dict]
    focus: str
    keywords: str          # regex; new lines with no match are ignored without a model call. Keep it permissive.
    events_schema: type[BaseModel]
    dedupe_key: Callable[[BaseModel], tuple]
    # The digest (section 10) is a template: events are sorted, grouped under headings and written one per line
    digest_sort: Callable[[BaseModel], tuple]
    digest_group: Callable[[BaseModel], str]
    digest_line: Callable[[BaseModel], str]
    digest_note: str = ""


# Breaking changes first within each provider; everything else after, in this order
MODEL_CHANGE_ORDER = ["retirement", "deprecation", "pricing", "api_change", "rate_limit",
                      "new_model", "capability", "other"]


def model_change_line(e: ChangeEvent) -> str:
    model = f"**{e.model}** " if e.model else ""
    when = f" Effective {e.effective_date}." if e.effective_date else ""
    return f"{model}{e.change_type.replace('_', ' ')}: {e.summary}{when}"


MODELS = Domain(
    name="models",
    sources=MODEL_SOURCES,
    focus="AI model launches, deprecations and retirement dates, pricing, API/parameter changes, rate limits, capabilities",
    keywords=(r"deprecat|retir|sunset|discontinu|shut ?down|end of life|launch|releas|introduc|now available"
              r"|general(ly)? availab|\bGA\b|preview|beta|experimental|pric|\$\s?\d|per (1M|million)|tokens?"
              r"|rate limit|quota|context (window|length)|parameter|endpoint|\bAPI\b|support"
              r"|\b(claude|gemini|gemma|imagen|veo|gpt|o\d|mistral|codestral|magistral|devstral|pixtral|ministral)\b"),
    events_schema=ChangeEvents,
    dedupe_key=lambda e: (e.provider.lower(), (e.model or "").lower(), e.change_type),
    digest_sort=lambda e: (e.provider.lower(), MODEL_CHANGE_ORDER.index(e.change_type)),
    digest_group=lambda e: e.provider,
    digest_line=model_change_line,
)

## Triage classifier

Every model verdict in `TRIAGE_LOG` is a labelled example: the new lines, the check signals and whether the change
was meaningful. Once there are `CLASSIFIER_MIN_EXAMPLES` of them, a logistic regression (TF-IDF of the new lines +
the signals + the domain) learns to predict that label. It is only trusted in one direction: when it gives a change
less than `CLASSIFIER_SKIP_BELOW` chance of mattering, the model is skipped. Everything else still goes to the model,
which keeps adding to the log.

Before it is used, 5-fold cross-validation checks how many real changes it would have skipped. Above
`CLASSIFIER_MAX_MISS_RATE`, it stays off.

In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder

SIGNAL_COLUMNS = ["added_lines", "removed_lines", "text_change", "dom_change", "length_ratio"]
FEATURE_COLUMNS = ["domain", "added_text", *SIGNAL_COLUMNS]


def load_labelled_log() -> pd.DataFrame:
    """TRIAGE_LOG rows that have a verdict, one per distinct change (re-runs log the same change again)."""
    lines = _read(Path(TRIAGE_LOG)) or ""
    rows = [json.loads(line) for line in lines.splitlines() if line.strip()]
    rows = [{**r, "meaningful": r["verdict"]["meaningful"]} for r in rows if r.get("verdict")]
    log = pd.DataFrame(rows, columns=["time", "meaningful", *FEATURE_COLUMNS])
    return log.drop_duplicates(["domain", "added_text"], keep="last")


def train_triage_classifier() -> Pipeline | None:
    """Fit the classifier on TRIAGE_LOG, or return None if there is too little data or it would skip real changes."""
    log = load_labelled_log()
    labels = log["meaningful"].astype(bool)
    if len(log) < CLASSIFIER_MIN_EXAMPLES or min(labels.sum(), (~labels).sum()) < 5:
        print(f"Triage classifier off: {len(log)} distinct labelled changes in {TRIAGE_LOG}; it needs "
              f"{CLASSIFIER_MIN_EXAMPLES}, with at least 5 of each label.")
        return None

    classifier = make_pipeline(
        ColumnTransformer([
            ("domain", OneHotEncoder(handle_unknown="ignore"), ["domain"]),
            ("text", TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True), "added_text"),
            ("signals", "passthrough", SIGNAL_COLUMNS),
        ]),
        LogisticRegression(class_weight="balanced", max_iter=1000),
    )
    features = log[FEATURE_COLUMNS]
    chance = cross_val_predict(classifier, features, labels, cv=5, method="predict_proba")[:, 1]
    would_skip = chance < CLASSIFIER_SKIP_BELOW
    miss_rate = (would_skip & labels).sum() / labels.sum()
    print(f"Triage classifier, cross-validated on {len(log)} changes: would skip {would_skip.mean():.0%} of model "
          f"calls and miss {miss_rate:.1%} of real changes (limit {CLASSIFIER_MAX_MISS_RATE:.0%}).")
    if miss_rate > CLASSIFIER_MAX_MISS_RATE:
        print("  Too many misses: classifier off.")
        return None
    return classifier.fit(features, labels)


def p_meaningful(domain: "Domain", page: "Page") -> float | None:
    """The classifier's probability that this change matters, or None when the classifier is off."""
    if TRIAGE_CLASSIFIER is None:
        return None
    features = pd.DataFrame([{"domain": domain.name, "added_text": "\n".join(page.sig.added)[:MAX_DIFF_CHARS],
                              **page.sig.summary()}])[FEATURE_COLUMNS]
    return float(TRIAGE_CLASSIFIER.predict_proba(features)[0, 1])


TRIAGE_CLASSIFIER = train_triage_classifier()

## The pipeline

`run(domain)` goes step by step, and each step handles every page that is still open at once:

1. `check_page` fetches, normalizes and runs the change checks on all pages concurrently. Most pages are settled here.
2. The classifier settles changes it is sure don't matter.
3. Large and `drastic` changes are triaged by the cheap model (all in one `ask_all`).
4. Small changes and meaningful triaged ones are extracted (one more `ask_all`), then deduped.

Each status row says which step decided (`decided_by`: `etag`, `hash`, `broken`, `layout`, `keywords`, `classifier`
or `llm`), how long each stage took, and the model calls made for that source. The first run diffs against the seeded
baselines; run it again and every source should report no change with no model calls.

In [ ]:
from dataclasses import field

from IPython.display import Markdown, display


@dataclass(eq=False)
class Page:
    """One source on its way through the pipeline. `outcome` is the change-check result once the page is checked."""
    source: dict
    row: dict                            # the status-table row
    outcome: str = ""
    new_text: str = ""
    sig: Signals | None = None
    vs: str = ""                         # " vs seeded baseline" when compared with a baseline
    triage: Triage | None = None         # None when the change went straight to extraction
    events: list[BaseModel] = field(default_factory=list)

    def settle(self, status: str, decided_by: str) -> None:
        self.row.update(status=status, decided_by=decided_by)

    @property
    def open(self) -> bool:
        return not self.row["decided_by"]


def dedupe(events: list[BaseModel], key: Callable[[BaseModel], tuple]) -> list[BaseModel]:
    merged: dict[tuple, BaseModel] = {}
    for e in events:
        merged.setdefault(key(e), e)   # first source wins; production would merge citations
    return list(merged.values())


async def check_page(domain: Domain, src: dict) -> Page:
    """Fetch one page and run the change checks; the page stays open only if a model has to look at it."""
    name = src["name"]
    page = Page(src, {"source": name, "status": "", "decided_by": "", "events": 0, "raw_changed": False,
                      "fetch_s": 0.0, "parse_s": 0.0, "checks_s": 0.0})
    try:
        fetched = await fetch(src["url"], load_meta(domain.name, name))
    except httpx.HTTPStatusError as e:
        hint = " (bot-protected: needs Playwright or an Open Terms Archive feed)" if e.response.status_code == 403 else ""
        page.settle(f"HTTP {e.response.status_code}{hint}", "error")
        return page
    except httpx.HTTPError as e:
        page.settle(f"fetch error: {e.__class__.__name__}", "error")
        return page
    page.row["fetch_s"] = fetched.seconds
    if fetched.html is None:
        page.settle("no change (304 Not Modified)", "etag")
        return page

    started = time.perf_counter()
    page.new_text = await asyncio.to_thread(normalize, fetched.html)   # CPU-bound: keep other fetches moving
    page.row["parse_s"] = time.perf_counter() - started
    if not page.new_text:
        page.settle("empty after normalize (JS-rendered? try Playwright)", "error")
        return page

    old_text, old_html = load_snapshot(domain.name, name), load_snapshot_html(domain.name, name)
    if old_text is None:
        old_text, page.vs = load_baseline(domain.name, name), " vs seeded baseline"
    if old_text is None:
        save_snapshot(domain.name, name, page.new_text, fetched)
        page.settle("first snapshot saved", "first_snapshot")
        return page

    started = time.perf_counter()
    page.sig = change_signals(old_text, page.new_text, old_html, fetched.html)
    page.outcome = classify_change(page.sig, domain.keywords)
    page.row["checks_s"] = time.perf_counter() - started
    page.row["raw_changed"] = old_text != page.new_text   # what the old pipeline would have sent to triage
    if page.outcome != "broken":   # keep the last good copy until extraction works again
        save_snapshot(domain.name, name, page.new_text, fetched)

    vs, sig = page.vs, page.sig
    if page.outcome == "none":
        page.settle(f"no change{vs}", "hash")
    elif page.outcome == "broken":
        page.settle(f"extraction broke? text is {sig.length_ratio:.0%} of the previous copy; snapshot kept", "broken")
    elif page.outcome == "layout":
        page.settle(f"ignored{vs}: no new lines ({len(sig.removed)} removed or reordered)", "layout")
    elif page.outcome == "no_keywords":
        page.settle(f"ignored{vs}: {len(sig.added)} new lines, no tracked keywords", "keywords")
    return page


def goes_straight_to_extract(page: Page) -> bool:
    """Small, ordinary changes skip triage: extraction finding no events is the same answer, for one call less."""
    return page.outcome == "normal" and len("\n".join(page.sig.added)) <= DIRECT_EXTRACT_MAX_CHARS


async def triage_pages(domain: Domain, pages: list[Page]) -> None:
    """Ask the cheap model about every page; settle the ones whose change isn't meaningful."""
    requests = [redesign_request(domain, p) if p.outcome == "drastic" else triage_request(domain, p) for p in pages]
    for page, req, verdict in zip(pages, requests, await ask_all(requests)):
        log_verdict(domain.name, page, req.step, verdict.model_dump() if verdict else None)
        page.triage = verdict
        if not verdict or not verdict.meaningful:
            label = "redesign, no tracked facts changed; rebaselined" if page.outcome == "drastic" else "ignored"
            page.settle(f"{label}{page.vs}: {verdict.reason if verdict else 'no verdict'}", "llm")


async def extract_pages(domain: Domain, pages: list[Page]) -> None:
    requests = [extract_request(domain, p) for p in pages]
    for page, result in zip(pages, await ask_all(requests)):
        found = result.events if result else []
        if page.triage is None:   # skipped triage: extraction's answer is the verdict
            log_verdict(domain.name, page, "extract", {"meaningful": bool(found)} if result else None)
        page.events = keep_verbatim(found, page)
        if result is None:
            page.settle(f"extraction failed{page.vs}: no parseable answer", "llm")
        elif not found:
            page.settle(f"ignored{page.vs}: no tracked changes found", "llm")
        else:
            severity = f" ({page.triage.severity})" if page.triage else ""
            page.settle(f"changed{severity}{page.vs}", "llm")
            page.row["events"] = len(page.events)


def with_llm_totals(domain: Domain, row: dict) -> dict:
    calls = [c for c in LLM_CALLS if c["domain"] == domain.name and c["source"] == row["source"]]
    return {**row, "llm_calls": len(calls),
            "cached_calls": sum(c["via"] == "cache" for c in calls),
            "llm_s": sum(c["latency_s"] for c in calls),
            "tokens_in": sum(c["tokens_in"] for c in calls),
            "tokens_out": sum(c["tokens_out"] for c in calls),
            "cost_usd": sum(c["cost_usd"] for c in calls)}


async def run(domain: Domain) -> tuple[list[BaseModel], pd.DataFrame]:
    LLM_CALLS[:] = [c for c in LLM_CALLS if c["domain"] != domain.name]   # re-running a domain replaces its numbers
    started = time.perf_counter()

    pages = await asyncio.gather(*(check_page(domain, src) for src in domain.sources))

    for page in [p for p in pages if p.open]:
        chance = p_meaningful(domain, page)
        if chance is not None and chance < CLASSIFIER_SKIP_BELOW:
            page.settle(f"ignored{page.vs}: classifier gives it a {chance:.1%} chance of mattering", "classifier")

    await triage_pages(domain, [p for p in pages if p.open and not goes_straight_to_extract(p)])
    await extract_pages(domain, [p for p in pages if p.open])

    print(f"{domain.name}: {len(pages)} pages in {time.perf_counter() - started:.1f}s")
    status = pd.DataFrame([with_llm_totals(domain, p.row) for p in pages])
    status = status.round({"fetch_s": 2, "parse_s": 2, "checks_s": 3, "llm_s": 2, "cost_usd": 5})
    events = [e for p in pages for e in p.events]
    return dedupe(events, domain.dedupe_key), status


model_events, model_status = await run(MODELS)
model_status

## Models digest

The digest is a template over the extracted events (never the raw pages): sorted and grouped by the domain's
`digest_*` functions, one bullet per event, each linking its source. There is no model call, so the digest is instant
and free, and every fact in it comes from an event whose quote was found on the page. With no events the digest is
empty.

In [ ]:
from datetime import date


def digest(domain: Domain, events: list[BaseModel], title: str) -> str:
    if not events:
        return ""
    groups: dict[str, list[BaseModel]] = defaultdict(list)
    for event in sorted(events, key=domain.digest_sort):
        groups[domain.digest_group(event)].append(event)
    lines = [f"## {title}"]
    for group, group_events in groups.items():
        lines += ["", f"### {group}", ""]
        lines += [f"- {domain.digest_line(e)} [Source]({e.source_url})" for e in group_events]
    if domain.digest_note:
        lines += ["", domain.digest_note]
    return "\n".join(lines)


models_digest = digest(MODELS, model_events, "AI model changes")
display(Markdown(models_digest or "_No model changes to report._"))

## Legal subagent

Same `run()` and `digest()`: only the sources, schema, focus and digest layout differ.

In [ ]:
JURISDICTION_ORDER = ["EU", "UK", "US-federal", "US-state", "other"]
# Within a jurisdiction: rules first, then litigation and terms-of-service changes
INSTRUMENT_ORDER = ["legislation", "regulation", "guidance", "enforcement", "litigation", "terms_of_service", "other"]


def legal_line(e: LegalEvent) -> str:
    when = f" Effective {e.effective_date}." if e.effective_date else ""
    return f"**{e.title}** ({e.instrument.replace('_', ' ')}): {e.summary} *Impact:* {e.impact}{when}"


LEGAL = Domain(
    name="legal",
    sources=LEGAL_SOURCES,
    focus=("Legal and regulatory developments affecting companies that provide LLM services: terms of service and "
           "usage-policy changes, EU AI Act / GPAI obligations and guidance, US federal and state AI rules, "
           "enforcement actions, and lawsuits (copyright, privacy, consumer protection)"),
    keywords=(r"\b(shall|must|may not|prohibit|permit|allow|restrict|requir|obligat|comply|complian)"
              r"|effective|enter(s|ed)? into (force|application)|apply from|deadline|amend|updat|revis|replac"
              r"|regulation|directive|\bact\b|article \d+|annex|guideline|guidance|code of practice|template"
              r"|court|lawsuit|complaint|settlement|fine|penalt|enforce|consultation|signator"),
    events_schema=LegalEvents,
    dedupe_key=lambda e: (e.jurisdiction, _squash(e.title)),
    digest_sort=lambda e: (JURISDICTION_ORDER.index(e.jurisdiction), INSTRUMENT_ORDER.index(e.instrument)),
    digest_group=lambda e: e.jurisdiction,
    digest_line=legal_line,
    digest_note="_This is monitoring, not legal advice._",
)

legal_events, legal_status = await run(LEGAL)
display(legal_status)
legal_digest = digest(LEGAL, legal_events, "Legal & regulatory updates for LLM providers")
display(Markdown(legal_digest or "_No legal changes to report._"))

## Cost & latency

Where the time and money went, per source, per model and step, and per route (`live`, `batch` or `cache`), and what
the checks saved. The old pipeline sent every page whose text differed at all to triage, so "avoided" counts pages
whose raw text changed but were settled without a model call: by the checks, the classifier, or the cache. The saving
uses this run's measured mean triage latency and cost (falling back to `FALLBACK_TRIAGE_*` when there were no paid
triage calls). That is conservative, because the old triage used Flash on the full diff, not Flash-Lite on the new
lines.

In [ ]:
CALL_COLUMNS = ["domain", "source", "step", "model", "via", "latency_s", "tokens_in", "tokens_out", "cost_usd"]
calls = pd.DataFrame(LLM_CALLS, columns=CALL_COLUMNS)
status = pd.concat([model_status.assign(domain="models"), legal_status.assign(domain="legal")], ignore_index=True)

print("Per source")
display(status[["domain", "source", "decided_by", "fetch_s", "parse_s", "checks_s",
                "llm_calls", "cached_calls", "llm_s", "tokens_in", "tokens_out", "cost_usd"]])

print("Per model, step and route")
display(calls.groupby(["model", "step", "via"]).agg(
    calls=("model", "size"), mean_latency_s=("latency_s", "mean"), total_latency_s=("latency_s", "sum"),
    tokens_in=("tokens_in", "sum"), tokens_out=("tokens_out", "sum"), cost_usd=("cost_usd", "sum"),
).round({"mean_latency_s": 2, "total_latency_s": 2, "cost_usd": 5}))

paid_triage = calls[calls["step"].isin(["triage", "redesign"]) & (calls["via"] != "cache")]
if len(paid_triage):
    mean_s, mean_usd = paid_triage["latency_s"].mean(), paid_triage["cost_usd"].mean()
    basis = f"mean of {len(paid_triage)} measured triage calls"
else:
    mean_s, mean_usd, basis = FALLBACK_TRIAGE_SECONDS, FALLBACK_TRIAGE_USD, "fallback constants from the config cell"
# Pages whose text changed but that cost no model call: settled by the checks or the classifier, or cache hits only
avoided = status[status["raw_changed"] & (status["llm_calls"] == status["cached_calls"])]
print(f"Model calls avoided: {len(avoided)} of {int(status['raw_changed'].sum())} pages whose text changed")
print(f"Estimated saving: {len(avoided) * mean_s:.1f}s of model latency and ${len(avoided) * mean_usd:.5f} ({basis}); "
      f"the checks themselves took {status['checks_s'].sum():.3f}s in total")
print(f"Cache hits: {int((calls['via'] == 'cache').sum())} of {len(calls)} answered requests")
display(avoided[["domain", "source", "decided_by", "status"]])

## Save digest

The combined digest (with the charts above) is written to `output/digest-<date>.md` and rendered below.
If neither domain has anything to report, no file is written and `digest_path` is `None`.
Email delivery comes next.

In [ ]:
def save_digest(text: str) -> Path:
    path = Path(OUTPUT_DIR) / f"digest-{date.today().isoformat()}.md"
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text)
    print(f"saved {len(text):,} chars to {path}")
    return path


sections = [d for d in (models_digest, legal_digest) if d]
if sections:
    digest_path = save_digest("\n\n".join(sections))
    # Render the saved file; image links are rewritten to resolve from the notebook's directory
    display(Markdown(digest_path.read_text().replace("](chart-", f"]({OUTPUT_DIR}/chart-")))
else:
    digest_path = None
    print("No changes to report; no digest written.")